# Referee prepared-data audit

Summarizes split sizes and positive/negative counts for the four `encoded_presence` targets used by training. This is a data audit, not model evaluation. The held-out test manifest is read for counts only; the model-evaluation notebook uses validation by default.

Metrics in notebook 02 measure agreement with the encoded bits; they do not establish reviewed text/number detection quality.

In [ ]:
import json
from pathlib import Path

import pandas as pd

root_candidates = (Path.cwd(), *Path.cwd().parents)
REPOSITORY_ROOT = next((path for path in root_candidates if (path / 'models/referee/data/src/settings.py').is_file()), None)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root from the notebook working directory.')
DATA_ROOT = REPOSITORY_ROOT / 'models/referee/data/.local/data/train'
PREPARED_DIR = DATA_ROOT / 'prepared'
SUMMARY_PATH = DATA_ROOT / 'dataset-summary.json'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
summary['profile'], summary['available_splits']

In [ ]:
records = []
for split in ('train', 'validation', 'test'):
    manifest = PREPARED_DIR / 'manifests' / f'{split}.jsonl'
    if not manifest.exists():
        continue
    rows = [json.loads(line) for line in manifest.read_text(encoding='utf-8').splitlines() if line.strip()]
    if any(len(row.get('encoded_presence', [])) != 4 for row in rows):
        raise ValueError(f'{split} manifest does not have four encoded_presence bits.')
    for bit_index in range(4):
        values = [row['encoded_presence'][bit_index] for row in rows]
        records.append({
            'split': split, 'target': f'bit_{bit_index + 1}', 'rows': len(rows),
            'positive': sum(value == 1 for value in values),
            'negative': sum(value == 0 for value in values),
        })

support = pd.DataFrame(records)
support

## Interpreting the counts

These four targets match the current model head. Encoded-bit metrics check agreement with the training target; do not describe them as reviewed text/number detector quality. Average precision is unavailable for a bit if its selected split contains only one class.